# Phase 0 — Environment and Tooling

**Outcome:** a reproducible Python 3.15 project with uv, a working Jupyter kernel, and
the introspection habits used throughout the course.

| Unit | Topic |
| --- | --- |
| 0.1 | Python, the language and its implementations |
| 0.2 | Installing Python and virtual environments |
| 0.3 | Project management with uv |
| 0.4 | Jupyter and editor workflow (with Lab 0) |

**How to use this notebook**

- Select the **Python 3.15 (course)** kernel, then run the *Shared setup* cell first.
- Each unit starts with its own setup cell, so you can jump straight to any unit.
- *Predict the output* questions hide the answer in a collapsible block: decide first,
  then run the proof cell.
- Shell commands (`uv …`, `git …`) are shown in `bash` blocks for you to run in a
  terminal. The notebook only executes read-only commands such as `uv --version`.

## Shared setup

This cell runs first in every notebook of the course. It imports common modules and
defines three helpers used everywhere:

| Helper | What it does |
| --- | --- |
| `run_checks(test_cases)` | runs `(label, thunk, expected)` cases, prints PASS/FAIL |
| `show_size(label, obj)` | prints `sys.getsizeof(obj)`, type name and `id()` |
| `find_repo_root()` | finds the folder holding `pyproject.toml` |

`run_checks` catches every exception, so a failing check never stops Run All.

> **Note:** each unit re-imports what it needs, so you can run any unit on its own
> after this cell.

In [1]:
import inspect
import os
import platform
import sys
from collections.abc import Callable
from pathlib import Path

type TestCase = tuple[str, Callable[[], object], object]


def run_checks(test_cases: list[TestCase]) -> None:
    """Run (label, thunk, expected) cases and print PASS/FAIL for each. Never raises."""
    passed_count = 0
    for label, thunk, expected in test_cases:
        try:
            actual = thunk()
        except Exception as error:  # a check cell must never stop Run All
            print(f"FAIL  {label}: raised {type(error).__name__}: {error}")
            continue
        if actual == expected:
            passed_count += 1
            print(f"PASS  {label}")
        else:
            print(f"FAIL  {label}: expected {expected!r}, got {actual!r}")
    print(f"{passed_count}/{len(test_cases)} checks passed")


def show_size(label: str, obj: object) -> None:
    """Print the shallow size in bytes, the type and the id of an object."""
    print(f"{label}: {sys.getsizeof(obj)} bytes, type={type(obj).__name__}, id={id(obj)}")


def find_repo_root(start: Path | None = None) -> Path:
    """Walk up from start (default: the working directory) to the folder holding pyproject.toml."""
    current = (start or Path.cwd()).resolve()
    for folder in (current, *current.parents):
        if (folder / "pyproject.toml").exists():
            return folder
    raise FileNotFoundError("pyproject.toml not found above the working directory")


print(f"Python {platform.python_version()} ({sys.implementation.name}) on {platform.system()}")
print(f"Repository root: {find_repo_root().name}/")

Python 3.15.1 (cpython) on Linux
Repository root: python-course/


## Unit 0.1 — Python, the Language and Its Implementations

**Objectives**

- Separate *Python the language* (a specification) from *CPython the program* that runs it.
- Know the yearly release cycle and how long each version is supported.
- Ask a running interpreter exactly which version and implementation it is.

**JS bridge**

| JavaScript concept | Python equivalent | The difference that bites |
| --- | --- | --- |
| ECMAScript specification | The Python Language Reference | Python's reference is less formal; CPython's behaviour is the de-facto spec for edge cases |
| V8, SpiderMonkey, JavaScriptCore | CPython, PyPy, MicroPython, GraalPy | Almost everyone runs CPython; in JS the engine depends on the browser or runtime |
| `process.version` (Node) | `sys.version`, `sys.version_info` | `sys.version_info` is a tuple you can compare, not a string |
| `process.execPath` | `sys.executable` | Tells you *which* interpreter (and so which environment) runs your code |

### 0.1.1 Language vs implementation

*Python* is a language defined by the
[Python Language Reference](https://docs.python.org/3/reference/index.html) plus the
[standard library docs](https://docs.python.org/3/library/index.html). Programs that read
and execute Python source are **implementations**:

| Implementation | Written in | Execution strategy | Typical use |
| --- | --- | --- | --- |
| **CPython** | C | Compiles to bytecode, runs it in an interpreter loop (with an experimental JIT since 3.13) | The reference implementation; what `python.org` and `uv` install |
| **PyPy** | RPython (a Python subset) | Tracing JIT compiler | Long-running pure-Python code that needs speed |
| **MicroPython** | C | Compact bytecode VM | Microcontrollers (ESP32, Raspberry Pi Pico) |
| **GraalPy** | Java on GraalVM | JIT via the Truffle framework | Embedding Python in JVM applications |

The JavaScript parallel is exact: **ECMAScript** is the specification, and **V8**
(Chrome, Node, Deno), **SpiderMonkey** (Firefox) and **JavaScriptCore** (Safari, Bun) are
engines that implement it.

> **Note:** this course targets CPython. "Under the hood" sections describe CPython
> internals, which other implementations are free to do differently.

The cell below asks the running interpreter which implementation it is.

In [2]:
import platform
import sys

implementation_name = sys.implementation.name
print(f"{implementation_name=}")
print(f"{platform.python_implementation()=}")

implementation_name='cpython'
platform.python_implementation()='CPython'


#### Under the hood: `sys.implementation`

`sys.implementation` is a namespace object every implementation must provide
([PEP 421](https://peps.python.org/pep-0421/)). `vars()` exposes its attributes as a dict.
Notice `cache_tag`: it is the suffix CPython puts on files in `__pycache__`
(for example `module.cpython-315.pyc`), so two different interpreters never read each
other's bytecode. `hexversion` packs the version into one integer, which is handy for
fast comparisons in C extensions.

In [3]:
import sys

for attribute_name, attribute_value in vars(sys.implementation).items():
    print(f"{attribute_name:32} {attribute_value!r}")

print(f"{sys.hexversion=:#x}")

name                             'cpython'
cache_tag                        'cpython-315'
version                          sys.version_info(major=3, minor=15, micro=1, releaselevel='final', serial=0)
hexversion                       51315184
_multiarch                       'x86_64-linux-gnu'
supports_isolated_interpreters   True
sys.hexversion=0x30f01f0


### 0.1.2 Release cycle

Since [PEP 602](https://peps.python.org/pep-0602/), CPython ships **one feature release
every October**. Each release gets:

- about **2 years of bugfix releases** (3.15.1, 3.15.2, …), then
- **security-only releases** until **5 years** after the first release.

| Version | Released | Bugfix support ends | End of life |
| --- | --- | --- | --- |
| 3.12 | October 2023 | April 2025 (18 months under the earlier rule) | October 2028 |
| 3.13 | October 2024 | October 2026 | October 2029 |
| 3.14 | October 2025 | October 2027 | October 2030 |
| 3.15 | October 2026 | October 2028 | October 2031 |

The live table is on the [devguide versions page](https://devguide.python.org/versions/);
the 3.15 schedule is [PEP 790](https://peps.python.org/pep-0790/).

**Why this course targets 3.15.** It is the newest stable release, so it has the longest
support window and the newest features. Two that matter later in the course:

| Feature | PEP | Version label |
| --- | --- | --- |
| UTF-8 mode is on by default (`open()` without `encoding=` uses UTF-8 on every OS) | [PEP 686](https://peps.python.org/pep-0686/) | New in 3.15 |
| Explicit lazy imports with the `lazy import` statement | [PEP 810](https://peps.python.org/pep-0810/) | New in 3.15 |

The next cell computes the support windows from the PEP 602 rule instead of hard-coding
them.

In [4]:
def support_window(minor_version: int) -> tuple[int, int]:
    """Return (bugfix_end_year, end_of_life_year) for 3.<minor_version> under PEP 602."""
    release_year = 2023 + (minor_version - 12)  # 3.12 shipped in October 2023
    return release_year + 2, release_year + 5


for minor_version in range(12, 16):
    bugfix_end_year, end_of_life_year = support_window(minor_version)
    print(f"3.{minor_version}: bugfix until ~{bugfix_end_year}, security until {end_of_life_year}")

3.12: bugfix until ~2025, security until 2028
3.13: bugfix until ~2026, security until 2029
3.14: bugfix until ~2027, security until 2030
3.15: bugfix until ~2028, security until 2031


**New in 3.15 — UTF-8 mode by default.** `sys.flags.utf8_mode` is `1` on 3.15 even
without `-X utf8`. On older versions it is usually `0` (except where the locale is
`C`/`POSIX`). The cell is guarded so it prints a skip message on an older kernel.

In [5]:
import locale
import sys

if sys.version_info >= (3, 15):
    print(f"{sys.flags.utf8_mode=}")
    print(f"{locale.getpreferredencoding(False)=}")
else:
    print(f"Skipped: UTF-8 mode default needs Python 3.15, this is {sys.version_info[:2]}")

sys.flags.utf8_mode=1
locale.getpreferredencoding(False)='utf-8'


### 0.1.3 Inspecting the interpreter

Five attributes answer "what exactly is running my code?". Each gets its own cell.

| Expression | Returns | Use it for |
| --- | --- | --- |
| `sys.version` | human-readable string (version, build date, compiler) | logs and bug reports |
| `sys.version_info` | named tuple `(major, minor, micro, releaselevel, serial)` | version checks in code |
| `sys.implementation.name` | lower-case name, e.g. `'cpython'` | implementation-specific branches |
| `platform.python_implementation()` | display name, e.g. `'CPython'` | reports |
| `sys.executable` | absolute path of the interpreter binary | finding which environment is active |

`sys.version` is a free-form string: print it, but never parse it.

In [6]:
import sys

print(sys.version)

3.15.1 (main, Oct 21 2026, 17:48:50) [Clang 23.1.3 ]


`sys.version_info` is a named tuple. Index it, unpack it, or read fields by name.
Comparing it to a plain tuple is the idiomatic version check.

In [7]:
import sys

print(f"{sys.version_info=}")
print(f"{sys.version_info.major=}, {sys.version_info.minor=}, {sys.version_info.micro=}")
print(f"{sys.version_info >= (3, 12)=}")

sys.version_info=sys.version_info(major=3, minor=15, micro=1, releaselevel='final', serial=0)
sys.version_info.major=3, sys.version_info.minor=15, sys.version_info.micro=1
sys.version_info >= (3, 12)=True


`sys.implementation.name` is the machine-friendly identifier (always lower case).

In [8]:
import sys

print(f"{sys.implementation.name=}")

sys.implementation.name='cpython'


`platform.python_implementation()` returns the display name of the same thing.

In [9]:
import platform

print(f"{platform.python_implementation()=}")
print(f"{platform.python_version()=}")

platform.python_implementation()='CPython'
platform.python_version()='3.15.1'


`sys.executable` is the path of the interpreter binary. In this notebook it should point
inside the project's `.venv/`, which proves the kernel uses the project environment.
The exact path depends on your machine and OS.

In [10]:
import sys
from pathlib import Path

print(f"{sys.executable=}")
print(f"{'.venv' in Path(sys.executable).parts=}")

sys.executable='/home/sourish/Kikori/projects/Pythonlearning/python-course/.venv/bin/python'
'.venv' in Path(sys.executable).parts=True


#### Predict the output

**Predict the output.**

```python
import sys
print(sys.version_info >= (3, 15), (3, 9) < (3, 15), "3.9" < "3.15")
```

<details><summary>Answer</summary>

`True True False`. Tuples compare element by element as numbers, so `(3, 9) < (3, 15)`
is `True`. Strings compare character by character: `"9"` is greater than `"1"`, so
`"3.9" < "3.15"` is `False`. This is why you compare `sys.version_info`, never
`sys.version`.

</details>

Run the next cell to check your prediction.

In [11]:
import sys

print(sys.version_info >= (3, 15), (3, 9) < (3, 15), "3.9" < "3.15")

True True False


**Predict the output.**

```python
import sys
version_pair = sys.version_info[:2]
print(type(version_pair).__name__, version_pair)
```

<details><summary>Answer</summary>

`tuple (3, 15)`. Slicing the named tuple returns a plain `tuple`, not another
`version_info` object, so the field names are gone.

</details>

Run the next cell to check your prediction.

In [12]:
import sys

version_pair = sys.version_info[:2]
print(type(version_pair).__name__, version_pair)

tuple (3, 15)


### Recap — Unit 0.1

| Concept | One-line takeaway | JS equivalent |
| --- | --- | --- |
| Language vs implementation | Python is a spec; CPython is the C program that runs it | ECMAScript vs V8 |
| Other implementations | PyPy (JIT), MicroPython (devices), GraalPy (JVM) | SpiderMonkey, JavaScriptCore, QuickJS |
| Release cycle | One release every October, about 2 years of bugfixes, 5 years total | Node's even-numbered LTS lines |
| `sys.version_info` | Compare it to tuples for version checks | `process.versions.node` (a string) |
| `sys.implementation` | Name, cache tag and version of the running implementation | `process.release.name` |
| `sys.executable` | Which interpreter binary, so which environment | `process.execPath` |

**Further reading**

- [`sys` — System-specific parameters](https://docs.python.org/3/library/sys.html)
- [`platform` — Access to underlying platform data](https://docs.python.org/3/library/platform.html)
- [Status of Python versions (devguide)](https://devguide.python.org/versions/)
- [PEP 602 — Annual release cycle](https://peps.python.org/pep-0602/),
  [PEP 790 — Python 3.15 release schedule](https://peps.python.org/pep-0790/),
  [PEP 421 — `sys.implementation`](https://peps.python.org/pep-0421/)
- [What's new in Python 3.15](https://docs.python.org/3.15/whatsnew/3.15.html)

## Unit 0.2 — Installing Python and Virtual Environments

**Objectives**

- Install and pin a Python version per project with `uv`, never touching the OS Python.
- Know what is inside `.venv/` and what "activating" it really does.
- Detect from code whether a virtual environment is active and where imports come from.

**JS bridge**

| JavaScript concept | Python equivalent | The difference that bites |
| --- | --- | --- |
| `nvm install 22` | `uv python install 3.15` | uv downloads standalone builds; nothing is compiled or installed system-wide |
| `.nvmrc` | `.python-version` | uv reads it automatically for every `uv run`, no shell hook needed |
| `node_modules/` | `.venv/` | `.venv` also contains its own `python` launcher, not just packages |
| Module resolution walks up `node_modules` folders | Imports search the `sys.path` list in order | No nesting: one flat `site-packages` per environment, one version of each package |

The next cell is this unit's setup: it re-imports what the unit needs and locates the
repository root and the virtual environment, so the unit runs on its own.

In [13]:
import os
import shutil
import subprocess
import sys
from pathlib import Path


def find_repo_root(start: Path | None = None) -> Path:
    """Walk up from start (default: the working directory) to the folder holding pyproject.toml."""
    current = (start or Path.cwd()).resolve()
    for folder in (current, *current.parents):
        if (folder / "pyproject.toml").exists():
            return folder
    raise FileNotFoundError("pyproject.toml not found above the working directory")


repo_root = find_repo_root()
venv_dir = repo_root / ".venv"
venv_scripts_dir = venv_dir / ("Scripts" if os.name == "nt" else "bin")
print(f"{repo_root.name=}, {venv_dir.exists()=}, {venv_scripts_dir.name=}")

repo_root.name='python-course', venv_dir.exists()=True, venv_scripts_dir.name='bin'


### 0.2.1 Installing with uv

[uv](https://docs.astral.sh/uv/) manages Python versions as well as packages. These
commands are run in a terminal (shown here, not executed from the notebook):

```bash
uv python install 3.15        # download a standalone CPython 3.15 build
uv python list                # every version uv knows: installed and downloadable
uv python pin 3.15            # write .python-version in the current project
```

`uv python pin` writes a one-line `.python-version` file. Every later `uv run`,
`uv sync` or `uv venv` in this folder uses that version.

Only **read-only** commands are executed from the notebook. The cell checks that `uv` is
on `PATH` first (`shutil.which`) and prints a skip message if it is not.

In [14]:
def run_read_only(command: list[str]) -> str:
    """Run a read-only command and return its output, or a skip message if missing."""
    if shutil.which(command[0]) is None:
        return f"Skipped: {command[0]!r} is not on PATH"
    no_color_environment = {**os.environ, "NO_COLOR": "1"}  # plain text, no ANSI codes
    completed = subprocess.run(
        command, capture_output=True, text=True, env=no_color_environment, check=False
    )
    return (completed.stdout or completed.stderr).strip()


print(run_read_only(["uv", "--version"]))
print(run_read_only(["uv", "python", "list", "--only-installed"]))

uv 0.13.5 (x86_64-unknown-linux-gnu)
cpython-3.15.1-linux-x86_64-gnu                 /home/sourish/.local/bin/python3.15 -> /home/sourish/.local/share/uv/python/cpython-3.15-linux-x86_64-gnu/bin/python3.15
cpython-3.15.1-linux-x86_64-gnu                 /home/sourish/.local/share/uv/python/cpython-3.15-linux-x86_64-gnu/bin/python3.15
cpython-3.15.1+freethreaded-linux-x86_64-gnu    /home/sourish/.local/bin/python3.15t -> /home/sourish/.local/share/uv/python/cpython-3.15+freethreaded-linux-x86_64-gnu/bin/python3.15t
cpython-3.15.1+freethreaded-linux-x86_64-gnu    /home/sourish/.local/share/uv/python/cpython-3.15+freethreaded-linux-x86_64-gnu/bin/python3.15t
cpython-3.14.6-linux-x86_64-gnu                 /usr/bin/python3.14
cpython-3.14.6-linux-x86_64-gnu                 /usr/bin/python3 -> python3.14
cpython-3.14.6-linux-x86_64-gnu                 /usr/bin/python -> python3
cpython-3.13.14-linux-x86_64-gnu                /usr/bin/python3.13


The pin itself is just text. Reading it shows exactly what `uv` will use.

In [15]:
pinned_version = (repo_root / ".python-version").read_text(encoding="utf-8").strip()
print(f"{pinned_version=}")
print(f"{sys.version_info[:2]=}")

pinned_version='3.15'
sys.version_info[:2]=(3, 15)


**Why never use the OS Python**

| Problem | What happens |
| --- | --- |
| The OS depends on it | Linux tools (package managers, desktop utilities) import system packages; upgrading or removing them can break the OS |
| You do not choose the version | It is whatever your distribution ships, and it changes on OS upgrades |
| Installs need `sudo` and collide | `pip install` into it mixes your packages with the OS's; modern distros block it ([PEP 668](https://peps.python.org/pep-0668/) "externally managed") |
| Not reproducible | Teammates and CI have different versions |

The cell compares the interpreter running this notebook with the first `python3` on a
`PATH` that excludes the virtual environment (usually the OS one). Paths vary by machine.

In [16]:
path_without_venv = os.pathsep.join(
    folder
    for folder in os.environ.get("PATH", "").split(os.pathsep)
    if Path(folder).resolve() != venv_scripts_dir.resolve()
)
os_python = shutil.which("python3", path=path_without_venv)
print(f"{sys.executable=}")
print(f"{os_python=}")

sys.executable='/home/sourish/Kikori/projects/Pythonlearning/python-course/.venv/bin/python'
os_python='/usr/bin/python3'


### 0.2.2 Virtual environments

A virtual environment is a folder that makes an existing Python installation behave as
if it had its own, separate `site-packages`. `uv sync` creates `.venv/` automatically.

| Item | Purpose |
| --- | --- |
| `pyvenv.cfg` | Marks the folder as a venv; `home` points at the base interpreter |
| `bin/` (`Scripts\` on Windows) | `python` (a link to the base interpreter), `activate` scripts, console scripts such as `jupyter` and `ruff` |
| `lib/python3.15/site-packages/` | Every third-party package installed into this environment |

First, `pyvenv.cfg`. Notice `include-system-site-packages = false`: the OS's packages are
invisible here.

In [17]:
print((venv_dir / "pyvenv.cfg").read_text(encoding="utf-8"))

home = /home/sourish/.local/share/uv/python/cpython-3.15-linux-x86_64-gnu/bin
implementation = CPython
uv = 0.13.5
version_info = 3.15
include-system-site-packages = false
prompt = python-course



The `bin/` folder: `python*` entries are links to the base interpreter; the rest are
console scripts created by installed packages (the equivalent of `node_modules/.bin`).
The cell lists the first 15 entries and the total.

In [18]:
bin_entries = sorted(entry.name for entry in venv_scripts_dir.iterdir())
python_entries = [name for name in bin_entries if name.startswith("python")]
print(f"{len(bin_entries)=}")
print(f"{python_entries=}")
print(f"{bin_entries[:15]=}")
print(f"{(venv_scripts_dir / 'python').resolve()=}")

len(bin_entries)=50
python_entries=['python', 'python3', 'python3.15']
bin_entries[:15]=['activate', 'activate.bat', 'activate.csh', 'activate.fish', 'activate.nu', 'activate.ps1', 'activate.xsh', 'activate_this.py', 'cffi-gen-src', 'deactivate.bat', 'debugpy', 'debugpy-adapter', 'httpx', 'idna', 'ipython']
(venv_scripts_dir / 'python').resolve()=PosixPath('/home/sourish/.local/share/uv/python/cpython-3.15.1-linux-x86_64-gnu/bin/python3.15')


`site-packages` is where `uv add` puts packages. `sysconfig` reports its path, and
`importlib.metadata.distributions()` lists what is installed there.

In [19]:
import importlib.metadata
import sysconfig

site_packages_dir = Path(sysconfig.get_paths()["purelib"])
installed_names = sorted({dist.metadata["Name"] for dist in importlib.metadata.distributions()})
print(f"{site_packages_dir.relative_to(repo_root)=}")
print(f"{len(installed_names)=}")
print(f"{installed_names[:10]=}")

site_packages_dir.relative_to(repo_root)=PosixPath('.venv/lib/python3.15/site-packages')
len(installed_names)=95
installed_names[:10]=['Jinja2', 'MarkupSafe', 'PyYAML', 'Pygments', 'Send2Trash', 'anyio', 'argon2-cffi', 'argon2-cffi-bindings', 'arrow', 'asttokens']


**Activation only changes `PATH`.** `source .venv/bin/activate` prepends `.venv/bin` to
`PATH`, sets `VIRTUAL_ENV`, and changes the prompt (`PS1`). Nothing else. The cell
prints the three lines of the activate script that do this.

In [20]:
activate_script = (venv_scripts_dir / "activate").read_text(encoding="utf-8")
activation_prefixes = ("VIRTUAL_ENV='", 'PATH="$VIRTUAL_ENV', 'PS1="(')
for line in activate_script.splitlines():
    if line.strip().startswith(activation_prefixes):
        print(line.strip())

VIRTUAL_ENV='/home/sourish/Kikori/projects/Pythonlearning/python-course/.venv'
PATH="$VIRTUAL_ENV/bin:$PATH"
PS1="(${VIRTUAL_ENV_PROMPT}) ${PS1-}"


So activation is never required: running `.venv/bin/python` directly gives you the
environment, because the interpreter finds `pyvenv.cfg` next to itself. The cell runs it
with `VIRTUAL_ENV` removed and the venv removed from `PATH`. `uv run` works the same way.

In [21]:
clean_environment = {key: value for key, value in os.environ.items() if key != "VIRTUAL_ENV"}
clean_environment["PATH"] = path_without_venv
probe_code = "import sys; print(sys.prefix != sys.base_prefix, sys.prefix)"
completed = subprocess.run(
    [str(venv_scripts_dir / "python"), "-c", probe_code],
    capture_output=True,
    text=True,
    env=clean_environment,
    check=False,
)
print(f"in venv without activation: {completed.stdout.strip()}")

in venv without activation: True /home/sourish/Kikori/projects/Pythonlearning/python-course/.venv


#### Under the hood: how `site` finds `pyvenv.cfg`

At startup CPython looks for `pyvenv.cfg` next to the executable or one folder up. If it
finds one, it sets `sys.prefix` to the venv folder, and the `site` module (imported
automatically unless `-S` is passed) adds the venv's `site-packages` to `sys.path`.
Because `include-system-site-packages = false`, the base installation's site-packages
and the per-user site directory are skipped.

In [22]:
import site

print(f"{sys.flags.no_site=}")
print(f"{site.ENABLE_USER_SITE=}")
for site_packages_path in site.getsitepackages():
    print(f"site-packages: {Path(site_packages_path).relative_to(repo_root)}")

sys.flags.no_site=0
site.ENABLE_USER_SITE=False
site-packages: .venv/lib/python3.15/site-packages


### 0.2.3 Detecting the environment

Inside a virtual environment `sys.prefix` is the venv folder, while `sys.base_prefix`
is the installation it was created from. Outside a venv they are equal. This is the
reliable test; `VIRTUAL_ENV` is only set by activation, and a Jupyter kernel is often
launched without it.

In [23]:
def in_virtualenv() -> bool:
    """Return True when the running interpreter belongs to a virtual environment."""
    return sys.prefix != sys.base_prefix


print(f"{sys.prefix=}")
print(f"{sys.base_prefix=}")
print(f"{in_virtualenv()=}")
print(f"{os.environ.get('VIRTUAL_ENV')=}")

sys.prefix='/home/sourish/Kikori/projects/Pythonlearning/python-course/.venv'
sys.base_prefix='/home/sourish/.local/share/uv/python/cpython-3.15.1-linux-x86_64-gnu'
in_virtualenv()=True
os.environ.get('VIRTUAL_ENV')='/home/sourish/Kikori/projects/Pythonlearning/python-course/.venv'


`sys.path` is the ordered list of folders searched by `import`. The first match wins.
The cell labels each entry so you can see where the standard library ends and
third-party packages begin.

In [24]:
def label_path_entry(entry: str) -> str:
    """Describe what kind of folder a sys.path entry is."""
    if entry in ("", os.getcwd()):
        return "working directory (notebook folder)"
    if entry.endswith(".zip"):
        exists_note = "present" if Path(entry).exists() else "absent, kept for zipped stdlib"
        return f"standard library zip ({exists_note})"
    if entry.endswith("lib-dynload"):
        return "standard library C extensions"
    if "site-packages" in entry and entry.startswith(sys.prefix):
        return "virtual environment site-packages (third-party)"
    if entry.startswith(sys.base_prefix):
        return "standard library (pure Python)"
    return "other (PYTHONPATH, .pth file or IPython)"


for position, entry in enumerate(sys.path):
    print(f"[{position}] {label_path_entry(entry):48} {entry or repr(entry)}")

[0] standard library zip (absent, kept for zipped stdlib) /home/sourish/.local/share/uv/python/cpython-3.15.1-linux-x86_64-gnu/lib/python315.zip
[1] standard library (pure Python)                   /home/sourish/.local/share/uv/python/cpython-3.15.1-linux-x86_64-gnu/lib/python3.15
[2] standard library C extensions                    /home/sourish/.local/share/uv/python/cpython-3.15.1-linux-x86_64-gnu/lib/python3.15/lib-dynload
[3] working directory (notebook folder)              ''
[4] virtual environment site-packages (third-party)  /home/sourish/Kikori/projects/Pythonlearning/python-course/.venv/lib/python3.15/site-packages


#### Predict the output

**Predict the output.**

```python
import sys
print(sys.prefix == sys.base_prefix)
```

<details><summary>Answer</summary>

`False`, because this kernel runs from `.venv`. `sys.prefix` is the venv folder,
`sys.base_prefix` is uv's standalone CPython installation.

</details>

Run the next cell to check your prediction.

In [25]:
print(sys.prefix == sys.base_prefix)

False


**Predict the output.**

```python
import sys
from pathlib import Path
print(Path(sys.executable).parent.name, Path(sys.executable).parent.parent.name)
```

<details><summary>Answer</summary>

`bin .venv` on Linux and macOS; `Scripts .venv` on Windows. The interpreter lives in the
venv's scripts folder, so its grandparent is the venv itself.

</details>

Run the next cell to check your prediction.

In [26]:
print(Path(sys.executable).parent.name, Path(sys.executable).parent.parent.name)

bin .venv


### 0.2.4 JS bridge

| npm / Node | uv / Python | Note |
| --- | --- | --- |
| `node_modules/` per project | `.venv/` per project | Both are disposable and git-ignored; recreate with `npm ci` / `uv sync` |
| Nested `node_modules` allow two versions of one package | One flat `site-packages`; exactly one version of each package | Version conflicts must be resolved, not nested |
| Node finds packages by walking up the folder tree | Python searches `sys.path` in order | Running from a different folder does not change which packages load |
| `require.resolve("pkg")` | `importlib.util.find_spec("pkg").origin` | Shows the file an import would load |
| Node binary is global (or per nvm version) | `.venv/bin/python` is per project | The environment includes its own interpreter link |

The cell resolves where `nbformat` would be imported from without importing it,
the same question `require.resolve` answers.

In [27]:
import importlib.util

nbformat_spec = importlib.util.find_spec("nbformat")
nbformat_origin = Path(nbformat_spec.origin)
print(f"{nbformat_origin.relative_to(repo_root)=}")
print(f"{importlib.metadata.version('nbformat')=}")

nbformat_origin.relative_to(repo_root)=PosixPath('.venv/lib/python3.15/site-packages/nbformat/__init__.py')
importlib.metadata.version('nbformat')='5.11.1'


Like `node_modules`, a `.venv` is big. The cell adds up the size of every file in
`site-packages`. The number depends on the installed versions and the OS.

In [28]:
total_bytes = sum(
    file_path.stat().st_size for file_path in site_packages_dir.rglob("*") if file_path.is_file()
)
print(f"site-packages size: {total_bytes / 1_000_000:.0f} MB")

site-packages size: 140 MB


### Recap — Unit 0.2

| Concept | One-line takeaway | JS equivalent |
| --- | --- | --- |
| `uv python install` | Downloads standalone CPython builds; no OS changes | `nvm install` |
| `.python-version` | Pins the interpreter version for the project | `.nvmrc` |
| OS Python | Belongs to the OS; never install into it | System Node from `apt` |
| `.venv/` | Own `python` link, own `site-packages`, `pyvenv.cfg` marker | `node_modules/` |
| Activation | Only edits `PATH`, `VIRTUAL_ENV` and the prompt; optional | Adding `node_modules/.bin` to `PATH` |
| `sys.prefix != sys.base_prefix` | Reliable "am I in a venv?" test | — |
| `sys.path` | Ordered import search list; first match wins | Node's module resolution algorithm |

**Further reading**

- [`venv` — Creation of virtual environments](https://docs.python.org/3/library/venv.html)
- [`site` — Site-specific configuration hook](https://docs.python.org/3/library/site.html)
- [The module search path (`sys.path`)](https://docs.python.org/3/library/sys_path_init.html)
- [PEP 405 — Python virtual environments](https://peps.python.org/pep-0405/),
  [PEP 668 — Externally managed environments](https://peps.python.org/pep-0668/)
- [uv: installing Python](https://docs.astral.sh/uv/guides/install-python/)

## Unit 0.3 — Project Management with uv

**Objectives**

- Use the six everyday uv commands: `init`, `add`, `add --dev`, `remove`, `run`, `sync`.
- Read `pyproject.toml` and `uv.lock` and explain what each one is for.
- Translate every npm habit into its uv equivalent.

**JS bridge**

| JavaScript concept | Python equivalent | The difference that bites |
| --- | --- | --- |
| `package.json` | `pyproject.toml` | TOML, not JSON; standard fields are defined by PEP 621 |
| `devDependencies` | `[dependency-groups] dev` | Groups are a standard ([PEP 735](https://peps.python.org/pep-0735/)), not a uv invention |
| `package-lock.json` | `uv.lock` | One lock covers every OS and Python version the project supports |
| `scripts` in `package.json` | `uv run <command>` | There is no built-in script table; you run commands directly |

The next cell is this unit's setup.

In [29]:
import importlib.metadata
import os
import shutil
import subprocess
import tomllib
from pathlib import Path


def find_repo_root(start: Path | None = None) -> Path:
    """Walk up from start (default: the working directory) to the folder holding pyproject.toml."""
    current = (start or Path.cwd()).resolve()
    for folder in (current, *current.parents):
        if (folder / "pyproject.toml").exists():
            return folder
    raise FileNotFoundError("pyproject.toml not found above the working directory")


def run_read_only(command: list[str]) -> str:
    """Run a read-only command and return its output, or a skip message if missing."""
    if shutil.which(command[0]) is None:
        return f"Skipped: {command[0]!r} is not on PATH"
    no_color_environment = {**os.environ, "NO_COLOR": "1"}  # plain text, no ANSI codes
    completed = subprocess.run(
        command, capture_output=True, text=True, env=no_color_environment, check=False
    )
    return (completed.stdout or completed.stderr).strip()


repo_root = find_repo_root()
print(f"{repo_root.name=}")

repo_root.name='python-course'


### 0.3.1 Core commands

These change the project, so they are run in a terminal, never from a notebook:

```bash
uv init --bare --name python-course   # create pyproject.toml (--bare: no sample main.py)
uv add jupyterlab nbformat            # add runtime dependencies, update uv.lock, install
uv add --dev ruff                     # add to the "dev" dependency group
uv remove nbformat                    # remove a dependency, update lock and environment
uv run python script.py               # run inside the project environment (syncs first)
uv sync                               # make .venv match uv.lock exactly
```

| Command | Edits `pyproject.toml` | Edits `uv.lock` | Changes `.venv` |
| --- | --- | --- | --- |
| `uv init` | creates it | — | — |
| `uv add` / `uv add --dev` | yes | yes | yes |
| `uv remove` | yes | yes | yes |
| `uv run` | — | only if out of date | syncs before running |
| `uv sync` | — | only if out of date | yes (adds **and removes** packages) |

The result of `uv add` and `uv add --dev` is visible in `pyproject.toml`. The standard
library's `tomllib` (3.11+) reads it.

In [30]:
with (repo_root / "pyproject.toml").open("rb") as pyproject_file:
    pyproject = tomllib.load(pyproject_file)

print(f"{pyproject['project']['name']=}")
print(f"{pyproject['project']['requires-python']=}")
print("dependencies (uv add):")
for requirement in pyproject["project"]["dependencies"]:
    print(f"  {requirement}")
print(f"dev group (uv add --dev): {pyproject['dependency-groups']['dev']}")

pyproject['project']['name']='python-course'
pyproject['project']['requires-python']='>=3.15'
dependencies (uv add):
  ipykernel>=7.4.0
  jupyterlab>=4.6.4
  jupytext>=1.19.6
  nbconvert>=7.17.2
  nbformat>=5.11.1
dev group (uv add --dev): ['ruff>=0.17.3']


> **Note:** `uv add` writes a lower bound (`>=` the version it resolved). The exact
> installed version lives in the lockfile, which is the next section.

### 0.3.2 The lockfile

`uv.lock` records the **exact** version, source URL and hashes of every package in the
dependency tree, including transitive ones. It is generated by uv (never edit it by hand)
and committed to git, so `uv sync` installs the same bytes on every machine.

The cell parses it and shows the entry for `jupyterlab`.

In [31]:
with (repo_root / "uv.lock").open("rb") as lock_file:
    lockfile = tomllib.load(lock_file)

locked_packages = {package["name"]: package for package in lockfile["package"]}
jupyterlab_entry = locked_packages["jupyterlab"]
print(f"{lockfile['version']=}, {lockfile['requires-python']=}")
print(f"{len(locked_packages)=}")
print(f"{jupyterlab_entry['version']=}")
print(f"{jupyterlab_entry['source']=}")
print(f"{jupyterlab_entry['wheels'][0]['hash'][:20]=}...")

lockfile['version']=1, lockfile['requires-python']='>=3.15'
len(locked_packages)=99
jupyterlab_entry['version']='4.6.4'
jupyterlab_entry['source']={'registry': 'https://pypi.org/simple'}
jupyterlab_entry['wheels'][0]['hash'][:20]='sha256:15b13f991d398'...


The lock and the environment agree: for every direct dependency, the locked version is
the installed version.

In [32]:
for package_name in ["ipykernel", "jupyterlab", "jupytext", "nbconvert", "nbformat", "ruff"]:
    locked_version = locked_packages[package_name]["version"]
    installed_version = importlib.metadata.version(package_name)
    versions_match = locked_version == installed_version
    print(f"{package_name:12} {locked_version=:10} {installed_version=:10} {versions_match=}")

ipykernel    locked_version=7.4.0      installed_version=7.4.0      versions_match=True
jupyterlab   locked_version=4.6.4      installed_version=4.6.4      versions_match=True
jupytext     locked_version=1.19.6     installed_version=1.19.6     versions_match=True
nbconvert    locked_version=7.17.2     installed_version=7.17.2     versions_match=True
nbformat     locked_version=5.11.1     installed_version=5.11.1     versions_match=True


ruff         locked_version=0.17.3     installed_version=0.17.3     versions_match=True


`uv tree` draws the dependency tree from the lock. `--frozen` makes it read the lock as
is, without resolving or writing anything, so it is safe to run here.

In [33]:
print(run_read_only(["uv", "tree", "--frozen", "--depth", "1"]))

python-course v0.1.0
├── ipykernel v7.4.0
├── jupyterlab v4.6.4
├── jupytext v1.19.6
├── nbconvert v7.17.2
├── nbformat v5.11.1
└── ruff v0.17.3 (group: dev)


#### Under the hood: hashes and `uv sync --locked`

Every wheel in `uv.lock` has a `sha256` hash; uv refuses to install a download whose hash
differs. After installation, each package's `RECORD` file stores a hash of **every file**
it installed. `importlib.metadata` exposes them, so we can verify one file ourselves.
RECORD uses URL-safe base64 without padding instead of hex.

In CI, run `uv sync --locked`: it fails if `uv.lock` is out of date with
`pyproject.toml` instead of silently updating it (the `npm ci` behaviour).

In [34]:
import base64
import hashlib

nbformat_distribution = importlib.metadata.distribution("nbformat")
init_record = next(
    path for path in nbformat_distribution.files if str(path) == "nbformat/__init__.py"
)
file_digest = hashlib.sha256(init_record.locate().read_bytes()).digest()
computed_hash = base64.urlsafe_b64encode(file_digest).rstrip(b"=").decode()
print(f"{init_record.hash.mode=}")
print(f"{init_record.hash.value=}")
print(f"{computed_hash == init_record.hash.value=}")

init_record.hash.mode='sha256'
init_record.hash.value='5K_oeXayKTVJcdsLs8abfrd1ssfw8TSG7ZEEifWfFQk'
computed_hash == init_record.hash.value=True


#### Predict the output

**Predict the output.**

```python
print(locked_packages["nbformat"]["version"] == importlib.metadata.version("nbformat"))
```

<details><summary>Answer</summary>

`True`. `uv sync` (run implicitly by `uv add` and `uv run`) installs exactly the
locked versions.

</details>

Run the next cell to check your prediction.

In [35]:
print(locked_packages["nbformat"]["version"] == importlib.metadata.version("nbformat"))

True


**Predict the output.**

```python
runtime_names = [req.split(">")[0] for req in pyproject["project"]["dependencies"]]
print("ruff" in runtime_names, "ruff" in locked_packages)
```

<details><summary>Answer</summary>

`False True`. `uv add --dev` puts ruff in the `dev` dependency group, not in the runtime
dependencies, but the lockfile still pins it so every developer gets the same ruff.

</details>

Run the next cell to check your prediction.

In [36]:
runtime_names = [req.split(">")[0] for req in pyproject["project"]["dependencies"]]
print("ruff" in runtime_names, "ruff" in locked_packages)

False True


### 0.3.3 npm mapping table

| npm | uv | Notes |
| --- | --- | --- |
| `npm init` | `uv init` | `--bare` for only `pyproject.toml` |
| `npm install <pkg>` | `uv add <pkg>` | Updates `pyproject.toml`, `uv.lock` and `.venv` in one step |
| `npm install` (no args) | `uv sync` | Also removes packages that are not in the lock |
| `npm i -D <pkg>` | `uv add --dev <pkg>` | Goes to `[dependency-groups] dev` |
| `npm uninstall <pkg>` | `uv remove <pkg>` | |
| `npx <tool>` | `uv run <tool>` (project) / `uvx <tool>` (one-off) | `uvx` runs a tool in a throwaway environment |
| `npm run <script>` | `uv run <command>` | No script table; write the command |
| `npm ci` | `uv sync --locked` | Fails instead of updating a stale lock |
| `package-lock.json` | `uv.lock` | Commit both |

The cell checks that every uv subcommand used in the table is listed by `uv --help`
on this machine.

In [37]:
uv_help_text = run_read_only(["uv", "--help"])
if uv_help_text.startswith("Skipped"):
    print(uv_help_text)
else:
    listed_commands = {
        line.split()[0] for line in uv_help_text.splitlines() if line.startswith("  ")
    }
    for uv_command in ["init", "add", "remove", "sync", "run", "lock", "tree"]:
        print(f"uv {uv_command:7} available={uv_command in listed_commands}")

uv init    available=True
uv add     available=True
uv remove  available=True
uv sync    available=True
uv run     available=True
uv lock    available=True
uv tree    available=True


### Recap — Unit 0.3

| Concept | One-line takeaway | JS equivalent |
| --- | --- | --- |
| `pyproject.toml` | Project metadata, dependency ranges and tool config | `package.json` |
| `uv add` / `uv add --dev` | Add a runtime / dev dependency, lock and install | `npm install` / `npm i -D` |
| `uv run` | Run a command in the synced project environment | `npx`, `npm run` |
| `uv sync` | Make `.venv` match the lock exactly | `npm install` |
| `uv.lock` | Exact versions and hashes for every OS; commit it | `package-lock.json` |
| `uv sync --locked` | Reproducible CI installs that fail on a stale lock | `npm ci` |
| RECORD hashes | Every installed file is hashed and verifiable | `integrity` field in the lock |

**Further reading**

- [uv: working on projects](https://docs.astral.sh/uv/guides/projects/)
- [uv: locking and syncing](https://docs.astral.sh/uv/concepts/projects/sync/)
- [`tomllib` — Parse TOML files](https://docs.python.org/3/library/tomllib.html),
  [`importlib.metadata`](https://docs.python.org/3/library/importlib.metadata.html)
- [PEP 621 — Project metadata in `pyproject.toml`](https://peps.python.org/pep-0621/),
  [PEP 735 — Dependency groups](https://peps.python.org/pep-0735/),
  [PEP 751 — A standard lockfile format](https://peps.python.org/pep-0751/)

## Unit 0.4 — Jupyter and Editor Workflow

**Objectives**

- Explain how the Jupyter server, the kernel and the `.ipynb` file fit together.
- Avoid hidden-state bugs caused by running cells out of order.
- Use the IPython magics you will rely on for the rest of the course.
- Treat notebooks as data: read them with `nbformat`, convert them with `jupytext`.
- Introspect any object with `help`, `dir`, `type`, `id`, `vars` and `inspect`.

**JS bridge**

| JavaScript concept | Python equivalent | The difference that bites |
| --- | --- | --- |
| Browser DevTools console / Node REPL | Jupyter kernel (IPython) | State persists across cells until you restart the kernel |
| Observable / RunKit notebooks | Jupyter notebooks | Cells run in the order *you* click, not top to bottom |
| `console.dir(obj)` | `dir(obj)`, `vars(obj)` | `vars` shows only the instance's own attributes |
| `Function.prototype.toString()` | `inspect.getsource(func)` | Reads the source file; fails for built-ins written in C |
| Prettier / ESLint | ruff format / ruff check | One tool does both |

The next cell is this unit's setup. It also creates `notebooks/_scratch/`
(git-ignored) for files written by the demos.

In [38]:
import inspect
import json
import sys
from collections import Counter
from pathlib import Path

from IPython import get_ipython


def find_repo_root(start: Path | None = None) -> Path:
    """Walk up from start (default: the working directory) to the folder holding pyproject.toml."""
    current = (start or Path.cwd()).resolve()
    for folder in (current, *current.parents):
        if (folder / "pyproject.toml").exists():
            return folder
    raise FileNotFoundError("pyproject.toml not found above the working directory")


repo_root = find_repo_root()
this_notebook_path = repo_root / "notebooks" / "phase0_environment_setup.ipynb"
scratch_dir = repo_root / "notebooks" / "_scratch"
scratch_dir.mkdir(parents=True, exist_ok=True)
print(f"{this_notebook_path.exists()=}, {scratch_dir.relative_to(repo_root)=}")

this_notebook_path.exists()=True, scratch_dir.relative_to(repo_root)=PosixPath('notebooks/_scratch')


### 0.4.1 Jupyter architecture

Three separate pieces cooperate:

| Piece | What it is | In this project |
| --- | --- | --- |
| **Front end** | The UI that edits cells and shows outputs | JupyterLab in a browser, or VS Code |
| **Jupyter server** | A web server that manages files and starts kernels (JupyterLab only; VS Code starts kernels itself) | `uv run jupyter lab` |
| **Kernel** | A separate process that executes code and sends results back over ZeroMQ | `.venv/bin/python -m ipykernel_launcher` |
| **Notebook file** | JSON on disk: cells, outputs, metadata | `notebooks/*.ipynb` |

```mermaid
flowchart LR
    frontend["Front end (JupyterLab / VS Code)"] -- "execute_request" --> kernel["Kernel (.venv python + ipykernel)"]
    kernel -- "stream / execute_result" --> frontend
    frontend -- "save / load" --> notebook_file["notebook.ipynb (JSON)"]
```

<details><summary>ASCII fallback</summary>

```text
+----------------------+   execute_request    +-------------------------+
| Front end            | -------------------> | Kernel process          |
| (JupyterLab/VS Code) | <------------------- | .venv python + ipykernel|
+----------------------+   outputs (ZeroMQ)   +-------------------------+
           |
           | save / load
           v
   notebook.ipynb (JSON)
```

</details>

A **kernelspec** tells front ends how to start a kernel. This project registered one
(run once in a terminal):

```bash
uv run python -m ipykernel install --user --name python-course --display-name "Python 3.15 (course)"
```

The cell reads that kernelspec back and checks that it starts this project's
interpreter.

In [39]:
from jupyter_client.kernelspec import KernelSpecManager, NoSuchKernel

try:
    course_kernel_spec = KernelSpecManager().get_kernel_spec("python-course")
except NoSuchKernel as error:
    print(f"Skipped: kernel not registered ({type(error).__name__})")
else:
    print(f"{course_kernel_spec.display_name=}")
    print(f"{course_kernel_spec.argv[1:4]=}")
    print(f"{Path(course_kernel_spec.argv[0]) == Path(sys.executable)=}")
print(f"{type(get_ipython()).__name__=}")

course_kernel_spec.display_name='Python 3.15 (course)'
course_kernel_spec.argv[1:4]=['-Xfrozen_modules=off', '-m', 'ipykernel_launcher']
Path(course_kernel_spec.argv[0]) == Path(sys.executable)=True
type(get_ipython()).__name__='ZMQInteractiveShell'


### 0.4.2 Execution discipline

Every executed cell gets a number in the **execution counter** (`[7]`). The kernel keeps
every variable alive until it restarts, so the *order you ran cells in* is what matters,
not their order on the page. IPython also keeps history: `In` (inputs), `Out`
(expression results) and `_` (the last result).

The cell reads the counter and the history through the shell object: `In[-1]` is the
source of the cell that is running right now.

In [40]:
ipython_shell = get_ipython()
print(f"{ipython_shell.execution_count=}")
print(f"{len(ipython_shell.user_ns['In'])=}")
print(f"{ipython_shell.user_ns['In'][-1].splitlines()[0]=}")

ipython_shell.execution_count=41
len(ipython_shell.user_ns['In'])=41
ipython_shell.user_ns['In'][-1].splitlines()[0]='ipython_shell = get_ipython()'


**Out-of-order hazard.** Imagine these three cells are far apart in a long notebook.
The function reads the global `tax_rate` *at call time*, so its result depends on which
cell ran last, not on what you see above it.

In [41]:
tax_rate = 0.18
print(f"{tax_rate=}")

tax_rate=0.18


In [42]:
def total_with_tax(price_in_rupees: float) -> float:
    """Return the price including the current global tax rate."""
    return round(price_in_rupees * (1 + tax_rate), 2)


print(f"{total_with_tax(100)=}")

total_with_tax(100)=118.0


In [43]:
tax_rate = 0.05  # a cell further down changes the global...
print(f"{total_with_tax(100)=}")  # ...and the function's answer changes with it

total_with_tax(100)=105.0


> **Warning:** a notebook that only works in the order you happened to run it is broken.
> Before saving or committing, use **Restart & Run All** (JupyterLab: *Kernel* menu;
> VS Code: *Restart* then *Run All*). This course verifies every notebook the same way:
> `jupyter nbconvert --execute` runs all cells top to bottom in a fresh kernel.

#### Predict the output

**Predict the output.**

```python
# cell 1
6 * 7
# cell 2
print(_ + 1)
```

<details><summary>Answer</summary>

`43`. When a cell ends in an expression, IPython displays it and stores it in `_`
(and in `Out[n]`). The next cell reads it back. `__` and `___` hold the two results
before that. (`_` is created by IPython at run time, so the proof cell tells ruff not
to report it as undefined with `# noqa: F821`.)

</details>

Run the next two cells to check your prediction.

In [44]:
6 * 7

42

In [45]:
print(_ + 1)  # noqa: F821

43


**Predict the output.**

```python
# cell A (later deleted from the notebook, but it already ran)
greeting = "hello"
# cell B
print(greeting)
# What does cell B print now, and what after Restart & Run All?
```

<details><summary>Answer</summary>

Now: `hello`, because the variable is still alive in the kernel. After Restart & Run
All: `NameError`, because no remaining cell defines `greeting`. The proof simulates the
restart with `del`.

</details>

Run the next cell to check your prediction.

In [46]:
greeting = "hello"
print(greeting)
del greeting  # what a restart does to every variable
try:
    print(greeting)
except NameError as error:
    print(f"{type(error).__name__}: {error}")

hello
NameError: name 'greeting' is not defined


### 0.4.3 Magics

Magics are IPython commands, not Python: `%name` applies to one line, `%%name` to the
whole cell. They only work in IPython and Jupyter.

| Magic | What it does |
| --- | --- |
| `%timeit stmt` | Runs a statement many times and reports mean ± std dev |
| `%%time` | Times one run of the whole cell (wall and CPU) |
| `%who` / `%whos` | Lists variables in the namespace / with type and value |
| `%load_ext autoreload` + `%autoreload 2` | Re-imports edited modules before each cell |
| `%%writefile path` | Writes the cell body to a file |
| `obj?` / `obj??` | Shows the docstring and signature / plus the source |

`%timeit` picks the loop count automatically; `-n` (loops) and `-r` (repeats) fix it so
the cell stays fast. Timings vary by machine.

In [47]:
%timeit -n 1000 -r 5 sum(range(1_000))

2.4 μs ± 148 ns per loop (mean ± std. dev. of 5 runs, 1,000 loops each)


`%%time` must be the first line of the cell and times everything below it once.

In [48]:
%%time
import random

random.seed(42)
random_numbers = [random.random() for _ in range(200_000)]
sorted_numbers = sorted(random_numbers)
print(f"{len(sorted_numbers)=}")

len(sorted_numbers)=200000
CPU times: user 31.4 ms, sys: 0 ns, total: 31.4 ms
Wall time: 31.1 ms


`%who` prints the names of every variable you defined (modules included). Pass a type
name to filter.

In [49]:
%who float

tax_rate	 


`%whos` adds the type and a preview of the value, as a table.

In [50]:
%whos float list

Variable          Type     Data/Info
------------------------------------
bin_entries       list     n=50
installed_names   list     n=95
python_entries    list     n=3
random_numbers    list     n=200000
runtime_names     list     n=5
sorted_numbers    list     n=200000
tax_rate          float    0.05


**Autoreload.** Normally `import` runs a module once and caches it in `sys.modules`;
editing the file later has no effect until you restart. `%autoreload 2` reloads every
changed module before each cell runs. The demo:

1. load the extension,
2. write a module with `%%writefile` into `notebooks/_scratch/`,
3. import it and call it,
4. rewrite the file and call again **without** re-importing.

`%%writefile {path}` expands Python variables in braces, so the module path comes from
`scratch_dir`.

In [51]:
%load_ext autoreload
%autoreload 2
greeting_module_path = scratch_dir / "greeting_tools.py"
if str(scratch_dir) not in sys.path:
    sys.path.insert(0, str(scratch_dir))

In [52]:
%%writefile {greeting_module_path}
def greet(name: str) -> str:
    """Return a greeting for name."""
    return f"Hello, {name}!"

Writing /home/sourish/Kikori/projects/Pythonlearning/python-course/notebooks/_scratch/greeting_tools.py

In [53]:
import greeting_tools

print(greeting_tools.greet("Asha"))

Hello, Asha!


Now rewrite the module on disk. No restart and no new `import`.

In [54]:
%%writefile {greeting_module_path}
def greet(name: str) -> str:
    """Return a warmer greeting for name."""
    return f"Namaste, {name}! Welcome to Python 3.15."

Overwriting /home/sourish/Kikori/projects/Pythonlearning/python-course/notebooks/_scratch/greeting_tools.py


Calling the same function again picks up the new code, because autoreload re-imported
the changed file before this cell ran.

In [55]:
print(greeting_tools.greet("Asha"))

Namaste, Asha! Welcome to Python 3.15.


**`?` and `??`.** `find_repo_root?` shows the signature and docstring; `find_repo_root??`
also shows the source. JupyterLab shows the result in a pager and VS Code in the output
area; when the notebook is executed headless (as this course's build does) the pager
text is not stored, so the cells may look empty in the saved file.

In [56]:
find_repo_root?

In [57]:
find_repo_root??

Under the hood `?` uses the `inspect` module, which works anywhere, not just in IPython:

In [58]:
print(f"{inspect.signature(find_repo_root)=}")
print(f"{inspect.getdoc(find_repo_root)=}")

inspect.signature(find_repo_root)=<Signature (start: pathlib.Path | None = None) -> pathlib.Path>
inspect.getdoc(find_repo_root)='Walk up from start (default: the working directory) to the folder holding pyproject.toml.'


**Why `uv add` beats `%pip install`.**

| | `%pip install pkg` in a cell | `uv add pkg` in a terminal |
| --- | --- | --- |
| Recorded in `pyproject.toml` | no | yes |
| Recorded in `uv.lock` | no | yes, with hashes |
| Survives `uv sync` | **no**: `uv sync` removes packages that are not in the lock | yes |
| Reproducible for teammates and CI | no | yes |

```bash
uv add httpx          # then restart the kernel so it sees the new package
```

> **Warning:** never put `%pip install` or `!uv add` in a course notebook. A notebook
> that installs packages changes the environment every time someone runs it.

### 0.4.4 Notebooks are JSON

An `.ipynb` file is a JSON document: top-level `metadata`, `nbformat` version, and a
list of `cells`, each with `cell_type`, `source`, and (for code) `outputs`.
`nbformat` reads and validates it. The cell loads **this notebook** and counts its cells
by type.

> **Note:** it reads the copy on disk, i.e. the last saved version, not unsaved edits.

In [59]:
import nbformat

if this_notebook_path.exists():
    this_notebook = nbformat.read(this_notebook_path, as_version=4)
    cell_type_counts = Counter(cell.cell_type for cell in this_notebook.cells)
    print(f"{this_notebook.nbformat=}, {this_notebook.nbformat_minor=}")
    print(f"{this_notebook.metadata.kernelspec.name=}")
    print(f"{len(this_notebook.cells)=}")
    print(f"{dict(cell_type_counts)=}")
else:
    print(f"Skipped: {this_notebook_path.name} not found on disk")

this_notebook.nbformat=4, this_notebook.nbformat_minor=5
this_notebook.metadata.kernelspec.name='python-course'
len(this_notebook.cells)=173
dict(cell_type_counts)={'markdown': 96, 'code': 77}


#### Under the hood: the raw JSON

Skipping `nbformat` and reading the file with `json` shows the storage format. Notice
that `source` is stored as a **list of lines**; `nbformat` joins it into one string for
you.

In [60]:
raw_notebook = json.loads(this_notebook_path.read_text(encoding="utf-8"))
first_code_cell = next(cell for cell in raw_notebook["cells"] if cell["cell_type"] == "code")
print(f"{sorted(raw_notebook)=}")
print(f"{sorted(first_code_cell)=}")
print(f"{type(first_code_cell['source']).__name__=}")
print(f"{first_code_cell['source'][:2]=}")

sorted(raw_notebook)=['cells', 'metadata', 'nbformat', 'nbformat_minor']
sorted(first_code_cell)=['cell_type', 'execution_count', 'id', 'metadata', 'outputs', 'source']
type(first_code_cell['source']).__name__='list'
first_code_cell['source'][:2]=['import inspect\n', 'import os\n']


**Predict the output.**

```python
print(type(this_notebook.cells[0].source).__name__, this_notebook.cells[0].cell_type)
```

<details><summary>Answer</summary>

`str markdown`. The first cell is the title (markdown), and `nbformat` returns `source`
as one string even though the file stores a list of lines.

</details>

Run the next cell to check your prediction.

In [61]:
print(type(this_notebook.cells[0].source).__name__, this_notebook.cells[0].cell_type)

str markdown


### 0.4.5 Editor setup

| | VS Code | PyCharm |
| --- | --- | --- |
| Python support | *Python* extension (`ms-python.python`) + **Pylance** (built on the **pyright** type checker) | Built in (its own type checker) |
| Notebooks | *Jupyter* extension (`ms-toolsai.jupyter`) | Built in (Professional edition) |
| Formatting and linting | *Ruff* extension (`charliermarsh.ruff`), format on save | Ruff plugin or File Watcher |
| Interpreter selection | *Python: Select Interpreter* → `.venv/bin/python`; notebook kernel picker → "Python 3.15 (course)" | *Settings → Python Interpreter* → existing uv environment |

The repository ships VS Code settings so the right interpreter and formatter are used
automatically. The cell prints them and the `[tool.ruff]` section they rely on.

In [62]:
import tomllib

vscode_dir = repo_root / ".vscode"
for settings_name in ["settings.json", "extensions.json"]:
    settings_path = vscode_dir / settings_name
    if settings_path.exists():
        print(f"{settings_name}: {json.loads(settings_path.read_text(encoding='utf-8'))}")
    else:
        print(f"Skipped: {settings_name} not found")

ruff_config = tomllib.loads((repo_root / "pyproject.toml").read_text(encoding="utf-8"))
print(f"[tool.ruff]: {ruff_config['tool']['ruff']}")

settings.json: {'python.defaultInterpreterPath': '${workspaceFolder}/.venv/bin/python', '[python]': {'editor.defaultFormatter': 'charliermarsh.ruff', 'editor.formatOnSave': True, 'editor.codeActionsOnSave': {'source.organizeImports': 'explicit'}}, 'notebook.defaultFormatter': 'charliermarsh.ruff', 'notebook.formatOnSave.enabled': True, 'jupyter.notebookFileRoot': '${fileDirname}'}
extensions.json: {'recommendations': ['ms-python.python', 'ms-toolsai.jupyter', 'charliermarsh.ruff']}
[tool.ruff]: {'line-length': 100, 'target-version': 'py315', 'extend-include': ['*.ipynb'], 'src': ['.', 'scripts'], 'lint': {'select': ['E', 'W', 'F', 'I', 'UP', 'B'], 'per-file-ignores': {'scripts/phase*/unit_*.py': ['E501'], 'scripts/phase*/project.py': ['E501'], 'notebooks/*.ipynb': ['UP036', 'F811']}}}


### 0.4.6 Jupytext

Notebook JSON makes noisy git diffs: outputs, execution counts and metadata change on
every run. [Jupytext](https://jupytext.readthedocs.io/) can **pair** an `.ipynb` with a
plain `.py` file in the *percent* format, where `# %%` starts a code cell and
`# %% [markdown]` a markdown cell. Reviewers read the `.py` diff.

```bash
uv run jupytext --set-formats ipynb,py:percent notebooks/example.ipynb   # pair
uv run jupytext --sync notebooks/example.ipynb                           # update the older side
```

> **Note:** this course does not pair its notebooks, because the build scripts in
> `scripts/` are already the readable source. The cell converts this notebook in memory
> only, to show the format.

In [63]:
import jupytext

percent_script = jupytext.writes(this_notebook, fmt="py:percent")
percent_lines = percent_script.splitlines()
first_cell_line = percent_lines.index("# %% [markdown]")
print("\n".join(percent_lines[first_cell_line : first_cell_line + 12]))
print(f"... {len(percent_lines)} lines in total")

# %% [markdown]
# # Phase 0 — Environment and Tooling
#
# **Outcome:** a reproducible Python 3.15 project with uv, a working Jupyter kernel, and
# the introspection habits used throughout the course.
#
# | Unit | Topic |
# | --- | --- |
# | 0.1 | Python, the language and its implementations |
# | 0.2 | Installing Python and virtual environments |
# | 0.3 | Project management with uv |
# | 0.4 | Jupyter and editor workflow (with Lab 0) |
... 1750 lines in total


The same notebook as percent script is much smaller than the JSON, mainly because
outputs and metadata are left out.

In [64]:
json_size = len(this_notebook_path.read_text(encoding="utf-8"))
print(f"{json_size=} characters, {len(percent_script)=} characters")

json_size=94167 characters, len(percent_script)=65037 characters


### 0.4.7 Introspection toolkit

Six built-ins answer "what is this object?". They work on everything, because in Python
everything is an object. This small class is the subject for each demo.

In [65]:
class ShoppingCart:
    """A minimal cart used to demonstrate introspection tools."""

    def __init__(self, owner_name: str) -> None:
        self.owner_name = owner_name
        self.item_prices_in_rupees: list[float] = []

    def add_item(self, price_in_rupees: float) -> None:
        """Add one item's price to the cart."""
        self.item_prices_in_rupees.append(price_in_rupees)

    def total(self) -> float:
        """Return the sum of all item prices."""
        return sum(self.item_prices_in_rupees)


cart = ShoppingCart("Asha")
cart.add_item(499.0)
cart.add_item(150.5)
print(f"{cart.total()=}")

cart.total()=649.5


`help(obj)` prints the signature and docstring (the same text `?` shows).

In [66]:
help(ShoppingCart.add_item)

Help on function add_item in module __main__:

add_item(self, price_in_rupees: float) -> None
    Add one item's price to the cart.



`dir(obj)` lists every attribute name, including inherited dunder methods. Filtering out
names that start with `_` leaves the public API.

In [67]:
all_attribute_names = dir(cart)
public_attribute_names = [name for name in all_attribute_names if not name.startswith("_")]
print(f"{len(all_attribute_names)=}")
print(f"{public_attribute_names=}")

len(all_attribute_names)=33
public_attribute_names=['add_item', 'item_prices_in_rupees', 'owner_name', 'total']


`type(obj)` returns the object's class. Classes are objects too, and their type is
`type`.

In [68]:
print(f"{type(cart)=}")
print(f"{type(cart.total)=}")
print(f"{type(ShoppingCart)=}")

type(cart)=<class '__main__.ShoppingCart'>
type(cart.total)=<class 'method'>
type(ShoppingCart)=<class 'type'>


`id(obj)` is an integer that is unique among objects alive at the same time (in CPython
it is the memory address). Compare ids, never their values: the number changes on every
run.

In [69]:
same_cart = cart
other_cart = ShoppingCart("Ravi")
print(f"{id(cart) == id(same_cart)=}")
print(f"{id(cart) == id(other_cart)=}")
print(f"{cart is same_cart=}")

id(cart) == id(same_cart)=True
id(cart) == id(other_cart)=False
cart is same_cart=True


`vars(obj)` returns the object's `__dict__`: only its **own** attributes. Methods live
on the class, so they do not appear.

In [70]:
print(f"{vars(cart)=}")
print(f"{'total' in vars(ShoppingCart)=}")

vars(cart)={'owner_name': 'Asha', 'item_prices_in_rupees': [499.0, 150.5]}
'total' in vars(ShoppingCart)=True


`inspect.getsource(obj)` returns the source code. It works for functions and classes
defined in files or notebook cells, and raises `TypeError` for built-ins written in C.

In [71]:
print(inspect.getsource(ShoppingCart.total))
try:
    inspect.getsource(len)
except TypeError as error:
    print(f"{type(error).__name__}: {error}")

    def total(self) -> float:
        """Return the sum of all item prices."""
        return sum(self.item_prices_in_rupees)

TypeError: module, class, method, function, traceback, frame, or code object was expected, got builtin_function_or_method


#### Predict the output

**Predict the output.**

```python
print(type(type), type(ShoppingCart) is type)
```

<details><summary>Answer</summary>

`<class 'type'> True`. `type` is its own type: it is the class of every class
(a *metaclass*, covered in Phase 2).

</details>

Run the next cell to check your prediction.

In [72]:
print(type(type), type(ShoppingCart) is type)

<class 'type'> True


**Predict the output.**

```python
empty_cart = ShoppingCart("Meera")
print(list(vars(empty_cart)))
```

<details><summary>Answer</summary>

`['owner_name', 'item_prices_in_rupees']`. Only attributes assigned on `self` in
`__init__` are in the instance `__dict__`, in assignment order; `add_item` and `total`
belong to the class.

</details>

Run the next cell to check your prediction.

In [73]:
empty_cart = ShoppingCart("Meera")
print(list(vars(empty_cart)))

['owner_name', 'item_prices_in_rupees']


### Lab 0

Write `environment_report()` that **returns** a dict describing the running interpreter
and also **prints** it, one `key: value` per line. Required keys:

| Key | Value |
| --- | --- |
| `python_version` | the version string, e.g. `'3.15.1'` (`platform.python_version()`) |
| `implementation` | e.g. `'CPython'` (`platform.python_implementation()`) |
| `executable` | `sys.executable` |
| `in_virtualenv` | `True` when a virtual environment is active |
| `jupyterlab_version` | installed JupyterLab version via `importlib.metadata.version` |

Replace the `raise` in the stub with your implementation, then run the check cell.
Until then every check prints FAIL with `NotImplementedError`; that is expected.

In [74]:
import importlib.metadata
import platform


def environment_report() -> dict[str, str | bool]:
    """Return and print a description of the running Python environment."""
    raise NotImplementedError("Lab 0: implement environment_report()")

The check cell compares your report with values computed independently. It silences
the report's printing (`redirect_stdout`) so the PASS/FAIL lines stay readable.

In [75]:
import contextlib
import io


def quiet_report() -> dict[str, str | bool]:
    """Call environment_report() without letting its prints reach the output."""
    with contextlib.redirect_stdout(io.StringIO()):
        return environment_report()


expected_keys = [
    "python_version",
    "implementation",
    "executable",
    "in_virtualenv",
    "jupyterlab_version",
]
run_checks(
    [
        ("returns a dict", lambda: isinstance(quiet_report(), dict), True),
        ("has exactly the five keys", lambda: sorted(quiet_report()), sorted(expected_keys)),
        ("python_version", lambda: quiet_report()["python_version"], platform.python_version()),
        (
            "implementation",
            lambda: quiet_report()["implementation"],
            platform.python_implementation(),
        ),
        ("executable", lambda: quiet_report()["executable"], sys.executable),
        (
            "in_virtualenv",
            lambda: quiet_report()["in_virtualenv"],
            sys.prefix != sys.base_prefix,
        ),
        (
            "jupyterlab_version",
            lambda: quiet_report()["jupyterlab_version"],
            importlib.metadata.version("jupyterlab"),
        ),
    ]
)

FAIL  returns a dict: raised NotImplementedError: Lab 0: implement environment_report()
FAIL  has exactly the five keys: raised NotImplementedError: Lab 0: implement environment_report()
FAIL  python_version: raised NotImplementedError: Lab 0: implement environment_report()
FAIL  implementation: raised NotImplementedError: Lab 0: implement environment_report()
FAIL  executable: raised NotImplementedError: Lab 0: implement environment_report()
FAIL  in_virtualenv: raised NotImplementedError: Lab 0: implement environment_report()
FAIL  jupyterlab_version: raised NotImplementedError: Lab 0: implement environment_report()
0/7 checks passed


### Recap — Unit 0.4

| Concept | One-line takeaway | JS equivalent |
| --- | --- | --- |
| Server / kernel / file | UI, execution process and JSON document are separate pieces | Browser tab, JS engine, saved `.js` |
| Kernelspec | Tells front ends which interpreter to start; registered with `ipykernel install` | Choosing the Node binary |
| Execution order | State lives in the kernel; only Restart & Run All proves a notebook works | A long-lived REPL session |
| Magics | `%timeit`, `%%time`, `%who(s)`, `%autoreload 2`, `%%writefile`, `?`/`??` | DevTools helpers like `console.time` |
| `uv add` over `%pip` | Dependencies belong in `pyproject.toml` and `uv.lock` | `npm install` over a global install |
| `nbformat` | Notebooks are JSON you can read and generate | `JSON.parse` on a file |
| Jupytext | Pairs `.ipynb` with a diff-friendly `.py:percent` script | — |
| Introspection | `help`, `dir`, `type`, `id`, `vars`, `inspect.getsource` | `console.dir`, `typeof`, `fn.toString()` |

**Further reading**

- [IPython built-in magic commands](https://ipython.readthedocs.io/en/stable/interactive/magics.html),
  [autoreload](https://ipython.readthedocs.io/en/stable/config/extensions/autoreload.html)
- [Jupyter architecture](https://docs.jupyter.org/en/latest/projects/architecture/content-architecture.html),
  [nbformat](https://nbformat.readthedocs.io/),
  [Jupytext formats](https://jupytext.readthedocs.io/en/latest/formats-scripts.html)
- [`inspect` — Inspect live objects](https://docs.python.org/3/library/inspect.html),
  [built-in functions (`dir`, `vars`, `id`, `type`, `help`)](https://docs.python.org/3/library/functions.html)
- [uv: using uv with Jupyter](https://docs.astral.sh/uv/guides/integration/jupyter/),
  [PEP 8 — Style guide](https://peps.python.org/pep-0008/)

## Phase 0 Deliverable

The course repository, set up with uv:

| Path | Purpose |
| --- | --- |
| `pyproject.toml` | Project metadata, dependencies, dev group and ruff config |
| `.python-version` | Pins Python 3.15 |
| `uv.lock` | Exact versions and hashes of every dependency |
| `notebooks/` | One notebook per phase |
| `projects/` | Code that outgrows a notebook |
| `exercises/solutions/` | Lab reference solutions (`phaseN_solutions.py`) |
| `scripts/` | Notebook build scripts (`nb_builder.py`, `phaseN/`) |
| `README.md` | Course description and setup instructions |

The first cell checks that every required path exists.

In [76]:
from pathlib import Path


def find_repo_root(start: Path | None = None) -> Path:
    """Walk up from start (default: the working directory) to the folder holding pyproject.toml."""
    current = (start or Path.cwd()).resolve()
    for folder in (current, *current.parents):
        if (folder / "pyproject.toml").exists():
            return folder
    raise FileNotFoundError("pyproject.toml not found above the working directory")


repo_root = find_repo_root()
required_paths = [
    "pyproject.toml",
    ".python-version",
    "uv.lock",
    "README.md",
    "notebooks",
    "projects",
    "exercises/solutions",
    "scripts",
]
for relative_path in required_paths:
    status = "ok" if (repo_root / relative_path).exists() else "MISSING"
    print(f"{status:8} {relative_path}")

ok       pyproject.toml
ok       .python-version
ok       uv.lock
ok       README.md
ok       notebooks
ok       projects
ok       exercises/solutions
ok       scripts


The second cell prints the repository tree. It asks git for the file list
(`git ls-files --cached --others --exclude-standard`), so everything git ignores,
such as `.venv/`, caches and `notebooks/_scratch/`, is left out. Without git it falls
back to walking the folders and skipping those names.

In [77]:
import os
import shutil
import subprocess

SKIPPED_FOLDERS = {".git", ".venv", "__pycache__", ".ipynb_checkpoints", ".ruff_cache", "_scratch"}


def list_repository_files(root: Path) -> list[str]:
    """List repository files, honouring git's ignore rules when git is available."""
    if shutil.which("git") and (root / ".git").exists():
        completed = subprocess.run(
            ["git", "ls-files", "--cached", "--others", "--exclude-standard"],
            cwd=root,
            capture_output=True,
            text=True,
            check=True,
        )
        return sorted(completed.stdout.splitlines())
    file_paths = []
    for folder, subfolders, file_names in os.walk(root):
        subfolders[:] = [name for name in subfolders if name not in SKIPPED_FOLDERS]
        file_paths += [str((Path(folder) / name).relative_to(root)) for name in file_names]
    return sorted(file_paths)


def print_tree(relative_paths: list[str]) -> None:
    """Print relative file paths as an indented tree with box-drawing connectors."""
    tree: dict[str, dict] = {}
    for relative_path in relative_paths:
        node = tree
        for part in Path(relative_path).parts:
            node = node.setdefault(part, {})

    def walk(node: dict[str, dict], prefix: str) -> None:
        names = sorted(node, key=lambda name: (not node[name], name))  # folders first
        for index, name in enumerate(names):
            is_last = index == len(names) - 1
            print(f"{prefix}{'└── ' if is_last else '├── '}{name}{'/' if node[name] else ''}")
            walk(node[name], prefix + ("    " if is_last else "│   "))

    print(".")
    walk(tree, "")


print_tree(list_repository_files(repo_root))

.
├── .vscode/
│   ├── extensions.json
│   └── settings.json
├── exercises/
│   └── solutions/
│       └── phase0_solutions.py
├── notebooks/
│   └── phase0_environment_setup.ipynb
├── projects/
│   └── .gitkeep
├── scripts/
│   ├── phase0/
│   │   ├── build.py
│   │   ├── unit_0_1.py
│   │   ├── unit_0_2.py
│   │   ├── unit_0_3.py
│   │   └── unit_0_4.py
│   └── nb_builder.py
├── .gitignore
├── .python-version
├── README.md
├── SYLLABUS.md
├── pyproject.toml
└── uv.lock
